# 16. NHL equivalency (NHLe)

Forecast, Phase 1, step 2a. A prospect's points in junior, college, the AHL or Europe say something about how he'll score in the NHL, but a point in the OHL isn't worth a point in the KHL. **NHLe** gives each league a **factor**: NHL points per game ≈ league points per game × factor. A factor of 0.40 means 1.00 point per game in that league is worth about 0.40 in the NHL.

Two ways to measure it, then a backtest on real rookie seasons to pick one:

- **Method A, next season:** players who played in league L one season and in the NHL the next. Compare their points per game.
- **Method B, same season:** players who played in two leagues in the same season (the published hockeystats.com approach), chained together to reach the NHL.

**Limits, up front:**
- **Points only.** Outside the NHL the profiles have games, goals, assists and points: no ice time, no expected goals, no WAR.
- **NHL-bound players only.** The profiles are NHL players' profiles, so every junior or European season here belongs to someone who reached the NHL. The factors describe NHL-bound players, not everyone in those leagues.

The code is in `src/nhle.py`. Data: the NHL player profiles already on disk (`data/bio_raw/`, no downloads). Results stay private (`data/career_totals.csv`, `data/nhle_factors.csv`, gitignored); nothing on the cards, the app or the README changes.

## 1. Setup

In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
from pathlib import Path

from nhle import (load_profiles, career_totals, league_inventory, next_season_pairs, direct_breakdown,
                  age_breakdown, method_a, same_season_links, method_b, compare_published, rookie_rows,
                  predict, age_multipliers, gp_rmse, gp_bias, ordering_checks, age_bucket,
                  LEAGUE_MAP, EXCLUDED, PUBLISHED, FIRST_SEASON, LAST_SEASON, SKIP_SEASONS,
                  MIN_GP_PAIR, MIN_DIRECT_PAIRS, MIN_GP_SAME, MIN_LINK, MAX_STEPS, AGE_BUCKETS)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 220)

DATA = Path("../data")
OUT = DATA                      # where the two CSV files are saved

def label(s):
    return f"{s}-{str(s + 1)[-2:]}"

# The leagues people usually ask about (the published list plus the ECHL), in a fixed order for tables
MAIN = list(PUBLISHED) + ["ECHL", "Czechia", "BCHL", "J20 Nationell"]

print(f"Factor window: {label(FIRST_SEASON)} to {label(LAST_SEASON)}, skipping "
      f"{', '.join(label(s) for s in sorted(SKIP_SEASONS))}")

Factor window: 2005-06 to 2025-26, skipping 2012-13, 2020-21


## 2. Career table

Every profile has `seasonTotals`: one row per season, team and league, for every level from minor hockey to the NHL. To turn that into one clean row per **skater, season and league**:

- **Regular season only** (`gameTypeId` 2). Playoff scoring has a different pace and small samples.
- **Skaters only.** Goalies don't score, so NHLe doesn't apply.
- **No tournaments or short events** (World Juniors, Olympics, Spengler Cup, and so on; full list in section 3). A 7-game tournament isn't a league season.
- **Old league names mapped to the current name** (section 3), so the SHL is one league from 2005 to 2025 even though the profiles call it "Sweden" before 2015.
- **Traded mid-season?** Games and points are summed over his teams in the same league.
- **No 2026-27.** The current season is only a few weeks old.
- **Age** = age on February 1 of the season, same as notebook 15.

In [2]:
raw = load_profiles(DATA / "bio_raw")
careers = career_totals(raw)

print(f"{raw['playerId'].nunique():,} profiles, {len(raw):,} season rows -> "
      f"{len(careers):,} skater-season-league rows for {careers['playerId'].nunique():,} skaters")
print(f"Seasons {label(careers['season'].min())} to {label(careers['season'].max())}")

# Checks: one row per skater, season and league; nothing excluded survived; no in-progress season
assert not careers.duplicated(["playerId", "season", "league"]).any()
assert not careers["league"].isin(EXCLUDED).any()
assert not careers["league"].isin(LEAGUE_MAP.keys()).any()
assert careers["season"].max() == 2025
print("Checks passed: no duplicates, no excluded events, every old name mapped, last season 2025-26")

careers.to_csv(OUT / "career_totals.csv", index=False)
print(f"Saved career_totals.csv ({len(careers):,} rows)")

3,400 profiles, 126,988 season rows -> 65,323 skater-season-league rows for 3,059 skaters
Seasons 1979-80 to 2025-26
Checks passed: no duplicates, no excluded events, every old name mapped, last season 2025-26
Saved career_totals.csv (65,323 rows)


## 3. League inventory and name mapping

There are hundreds of raw league names. The ones that matter here are the leagues NHL players come from. Three things to check before any math:

1. **Renamed leagues.** The profiles switched names around 2015 (Sweden became SHL, Finland became Liiga, Swiss and NLA became NL, and so on). Before 2015, NCAA players are stored under their **conference** (WCHA, Hockey East, CCHA, ECAC, NCHC, Big Ten, CHA), never "NCAA". The mapping below folds each into one name; the first and last seasons show the hand-off.
2. **Excluded events:** tournaments and short events, plus the US National Team Development Program's composite rows (U-17, U-18, USDP, NTDP, USA-S17...). Those mix USHL games with exhibitions and tournaments, and the same players also have a separate USHL row, which we keep.
3. **Anything short we missed:** any remaining league whose typical player-season is under 15 games gets printed, so nothing slips through unseen.

In [3]:
inv = league_inventory(raw)
print(f"{len(inv)} raw league names for skaters. The 40 with the most players:")
inv.head(40)

853 raw league names for skaters. The 40 with the most players:


,players,rows,first,last,median_gp,excluded,maps_to
league,,,,,,,
NHL,3059,22224,1983,2025,58.0,False,
AHL,2783,13645,1984,2025,49.0,False,
WJC-A,783,1127,1981,2015,6.0,True,
WC-A,687,1700,1991,2015,7.0,True,
OHL,655,2325,1982,2025,61.0,False,
WC,641,1200,1990,2025,8.0,True,
KHL,628,2442,2008,2025,47.0,False,
USHL,617,1145,1990,2023,47.0,False,
Champions HL,614,1193,2015,2025,6.0,True,


In [4]:
# The rename map: each old name, what it becomes, and the seasons it covers in the profiles
mapping = inv.loc[inv["maps_to"] != "", ["maps_to", "players", "first", "last"]].reset_index()
mapping = mapping.sort_values(["maps_to", "first"]).set_index(["maps_to", "league"])
print("Old name -> league we use (first and last season the profiles use that name):")
mapping

Old name -> league we use (first and last season the profiles use that name):


players  first  last
maps_to           league                             
Czechia           Czech               44   1987  2017
                  CzRep               96   1993  2015
DEL               Germany            114   1991  2015
HockeyAllsvenskan Sweden-2           131   1987  2015
                  Allsvenskan         22   1999  2014
ICEHL             Austria             64   1994  2015
                  EBEL                90   2011  2019
J20 Nationell     Swe-Jr.            167   1989  2014
                  SuperElit           27   1996  2014
                  J20 SuperElit       76   2015  2019
KHL               Russia              71   1996  2007
                  Rus-KHL             87   2008  2010
Liiga             Finland            191   1985  2015
MHL               Russia-Jr.          39   2009  2014
Mestis            Finland-2           59   1987  2014
NCAA              WCHA               204   1981  2014
                  CCHA               158   1987  2012
                  H-East             168   1989  2015
                  ECAC               108   1990  2015
                  CHA                  5   2001  2009
                  NCHC                35   2013  2014
                  Big Ten             26   2013  2014
NL                Swiss              174   1994  2015
                  NLA                 82   2009  2016
SHL               Sweden             266   1988  2014
SL                Swiss-2             19   2003  2014
                  NLB                 13   2007  2016

In [5]:
# Excluded events that appear in the profiles, and any league left with a median under 15 games
excl = inv[inv["excluded"]]
print(f"Excluded: {len(excl)} event names, {excl['rows'].sum():,} rows. Largest: "
      + ", ".join(excl.head(15).index))
short = inv[~inv["excluded"] & (inv["median_gp"] < 15) & (inv["rows"] >= 40)]
print("\nStill in, with a median under 15 games per player-season (40+ rows), for review:")
short[["players", "rows", "first", "last", "median_gp", "maps_to"]]

Excluded: 57 event names, 12,986 rows. Largest: WJC-A, WC-A, WC, Champions HL, WJC-20, WJ18-A, EHT, OG, WJC-18, Olympics, WHC-17, Spengler Cup, QC Int PW, WCup, U-18

Still in, with a median under 15 games per player-season (40+ rows), for review:


,players,rows,first,last,median_gp,maps_to
league,,,,,,
Swe-U18,140,332,1997,2015,11.0,
J18 Elit,89,178,2007,2019,14.0,
U16 SM,86,109,2001,2022,3.0,
J18 Allsvenskan,79,136,2006,2019,10.0,
Finland-2,59,100,1987,2014,5.0,Mestis
VHL,57,114,2010,2025,9.0,
U16 Div.1,53,93,2012,2022,9.0,
J20 Nationell,41,98,2002,2024,14.0,
Jr. C SM-sarja Q,41,46,1996,2012,9.0,


**Read the short list:** these are real leagues where NHL-bound players often appear only briefly (a Finnish second-tier loan, a few VHL games). They stay in. They rarely matter here, because a league needs 20+ games per player for method A and 8+ players per link for method B.

## 4. Method A: league L one season, the NHL the next

Take every player with **20+ games** in league L in season t and 20+ games in the NHL in season t+1. The factor is

> (sum of NHL points / sum of NHL games) / (sum of league L points / sum of league L games)

Sums, not an average of each player's ratio, so a 70-game season counts more than a 20-game one. A league gets a **direct** factor with **30+ pairs**. Seasons 2012-13 (lockout: NHL players in Europe) and 2020-21 (COVID: NHL prospects loaned to Europe) are skipped as either season of a pair.

**Who's in these pairs?** Only players who went straight from league L to a regular NHL job. From junior that's the elite (top picks at 18 or 19), so junior factors should come out high. Keep that in mind for section 7.

In [6]:
pairs = next_season_pairs(careers)
print(f"{len(pairs):,} next-season moves between two different leagues; "
      f"{(pairs['league_to'] == 'NHL').sum():,} of them into the NHL")
print(f"\nDirect factors ({MIN_DIRECT_PAIRS}+ pairs), forwards, defensemen and everyone:")
direct_breakdown(pairs).round(3)

10,430 next-season moves between two different leagues; 2,267 of them into the NHL

Direct factors (30+ pairs), forwards, defensemen and everyone:


,F factor,F pairs,D factor,D pairs,all factor,all pairs
league_from,,,,,,
KHL,0.669,50.0,0.688,26.0,0.687,76.0
SHL,0.498,49.0,0.713,26.0,0.552,75.0
AHL,0.448,1105.0,0.520,554.0,0.466,1659.0
NCAA,0.369,116.0,0.480,65.0,0.398,181.0
OHL,0.299,73.0,0.310,23.0,0.300,96.0
WHL,0.286,53.0,0.287,29.0,0.290,82.0
QMJHL,0.269,25.0,0.254,7.0,0.265,32.0


In [7]:
print("Direct factors by age in season t (blank = fewer than 20 pairs in that cell):")
age_breakdown(pairs).dropna(how="all").round(3)

Direct factors by age in season t (blank = fewer than 20 pairs in that cell):


bucket,<=18,19,20,21,22-24,25+
league_from,,,,,,
AHL,NaN,0.605,0.579,0.513,0.455,0.369
KHL,NaN,NaN,NaN,NaN,0.592,0.628
NCAA,NaN,0.451,0.396,0.373,0.328,NaN
OHL,0.324,0.277,NaN,NaN,NaN,NaN
WHL,0.282,0.302,NaN,NaN,NaN,NaN


**Chained method A (fills gaps only).** Many leagues rarely send players straight to the NHL: USHL players go to college, Swiss and German players often go to the SHL or the AHL first. For any league **without** 30 direct pairs:

- Build **next-season links** between any two leagues (L in season t, M in t+1, 20+ games in both, **8+ players** per link).
- Follow links until reaching a league that **has** a direct factor, then multiply. Example: USHL to NCAA, then NCAA's direct factor.
- **Fewest steps** win (at most 3, counting the final direct step). When several shortest paths exist, average them, each weighted by **its smallest player count**: a chain is only as reliable as its weakest link.
- A direct factor always beats a chain.

**Expect chained junior factors to run high.** Every next-season link includes a year of development: a USHL player at 18 is a better player at 19 in college. So USHL to NCAA to NHL credits the USHL with a year of growth. We don't correct for that yet; the published comparison and the backtest show how big it is. The median age at each link is in the paths below.

In [8]:
a = method_a(pairs)
print(f"Method A: {(a['path_type'] == 'direct').sum()} direct and {(a['path_type'] == 'chained').sum()} chained factors")
print("\nMain leagues:")
main_a = a.set_index("league").reindex(MAIN).dropna(subset=["factor"]).astype({"steps": int, "n_paths": int})
main_a[["factor", "pairs", "path_type", "steps", "n_paths"]].round(3)

Method A: 7 direct and 55 chained factors

Main leagues:


,factor,pairs,path_type,steps,n_paths
league,,,,,
KHL,0.687,76.0,direct,1,1
SHL,0.552,75.0,direct,1,1
NL,0.500,92.0,chained,2,3
Liiga,0.431,132.0,chained,2,3
AHL,0.466,1659.0,direct,1,1
DEL,0.426,33.0,chained,2,2
HockeyAllsvenskan,0.377,57.0,chained,2,1
NCAA,0.398,181.0,direct,1,1
OHL,0.300,96.0,direct,1,1


In [9]:
# The chains behind the main chained leagues: each link shows (players, median age at season t)
for lg, r in main_a[main_a["path_type"] == "chained"].iterrows():
    print(f"{lg}: {r['factor']:.3f}, {int(r['steps'])} steps")
    for p in r["paths"].split(" | "):
        print("    " + p)

NL: 0.500, 2 steps
    NL>AHL (31, age 27) x AHL direct (1659, age 22) = 0.494 [weight 31]
    NL>KHL (22, age 30) x KHL direct (76, age 24) = 0.500 [weight 22]
    NL>SHL (39, age 29) x SHL direct (75, age 21) = 0.505 [weight 39]
Liiga: 0.431, 2 steps
    Liiga>AHL (84, age 21) x AHL direct (1659, age 22) = 0.425 [weight 84]
    Liiga>KHL (26, age 27) x KHL direct (76, age 24) = 0.493 [weight 26]
    Liiga>SHL (22, age 27) x SHL direct (75, age 21) = 0.382 [weight 22]
DEL: 0.426, 2 steps
    DEL>AHL (25, age 27) x AHL direct (1659, age 22) = 0.419 [weight 25]
    DEL>KHL (8, age 27) x KHL direct (76, age 24) = 0.447 [weight 8]
HockeyAllsvenskan: 0.377, 2 steps
    HockeyAllsvenskan>SHL (57, age 20) x SHL direct (75, age 21) = 0.377 [weight 57]
USHL: 0.323, 2 steps
    USHL>NCAA (302, age 18) x NCAA direct (181, age 21) = 0.316 [weight 181]
    USHL>OHL (21, age 17) x OHL direct (96, age 19) = 0.385 [weight 21]
MHL: 0.225, 2 steps
    MHL>KHL (33, age 18) x KHL direct (76, age 24) = 0.

## 5. Method B: two leagues in the same season

Players with **5+ games in two leagues in the same season** (an AHL call-up, a mid-season move to Europe) give a direct comparison: same player, same year, two leagues. A link needs **8+ players**. Leagues without their own NHL link chain through others (fewest steps, the same weakest-link weights as above).

**Expect this to be thin with our data.** Juniors almost never play two leagues in one season, and the players who split a season between the NHL and another league are a special group: AHL call-ups, or NHL players who were struggling and got sent to Europe. The published version uses every player in every league, which we don't have.

In [10]:
links = same_season_links(careers)
print(f"{len(links)} same-season links with {MIN_LINK}+ players")
to_nhl = links[links["league_to"] == "NHL"].set_index("league_from")[["pairs", "factor", "median_age"]]
print("\nDirect links to the NHL:")
print(to_nhl.sort_values("pairs", ascending=False).round(3).to_string())

b = method_b(links)
print(f"\nMethod B: {(b['path_type'] == 'direct').sum()} direct and {(b['path_type'] == 'chained').sum()} chained factors")
main_b = b.set_index("league").reindex(MAIN).dropna(subset=["factor"]).astype({"steps": int, "n_paths": int})
main_b[["factor", "pairs", "path_type", "steps", "n_paths"]].round(3)

154 same-season links with 8+ players

Direct links to the NHL:
              pairs  factor  median_age
league_from                            
AHL          3694.0   0.382        23.0
NCAA           73.0   0.339        21.0
KHL            24.0   0.619        25.0
SHL            19.0   0.335        21.0
WHL            16.0   0.172        19.0
OHL            13.0   0.149        19.0
NL             10.0   0.292        28.0
ECHL            9.0   0.496        22.0

Method B: 8 direct and 24 chained factors


,factor,pairs,path_type,steps,n_paths
league,,,,,
KHL,0.619,24.0,direct,1,1
SHL,0.335,19.0,direct,1,1
NL,0.292,10.0,direct,1,1
Liiga,0.318,58.0,chained,2,3
AHL,0.382,3694.0,direct,1,1
DEL,0.301,25.0,chained,2,2
HockeyAllsvenskan,0.119,19.0,chained,2,1
NCAA,0.339,73.0,direct,1,1
OHL,0.149,13.0,direct,1,1


In [11]:
for lg, r in main_b[main_b["path_type"] == "chained"].iterrows():
    print(f"{lg}: {r['factor']:.3f}, {int(r['steps'])} steps")
    for p in r["paths"].split(" | "):
        print("    " + p)
missing_b = [lg for lg in PUBLISHED if lg not in main_b.index]
print("\nPublished leagues method B can't reach:", ", ".join(missing_b) or "none")

Liiga: 0.318, 2 steps
    Liiga>AHL (22, age 20) x AHL>NHL (3694, age 23) = 0.309 [weight 22]
    Liiga>KHL (17, age 27) x KHL>NHL (24, age 25) = 0.440 [weight 17]
    Liiga>SHL (23, age 27) x SHL>NHL (19, age 21) = 0.220 [weight 19]
DEL: 0.301, 2 steps
    DEL>AHL (12, age 28) x AHL>NHL (3694, age 23) = 0.233 [weight 12]
    DEL>KHL (13, age 29) x KHL>NHL (24, age 25) = 0.363 [weight 13]
HockeyAllsvenskan: 0.119, 2 steps
    HockeyAllsvenskan>SHL (51, age 19) x SHL>NHL (19, age 21) = 0.119 [weight 19]
MHL: 0.101, 2 steps
    MHL>KHL (55, age 18) x KHL>NHL (24, age 25) = 0.101 [weight 24]
QMJHL: 0.116, 2 steps
    QMJHL>AHL (16, age 19) x AHL>NHL (3694, age 23) = 0.116 [weight 16]
Czechia: 0.355, 2 steps
    Czechia>KHL (12, age 29) x KHL>NHL (24, age 25) = 0.355 [weight 12]
J20 Nationell: 0.058, 2 steps
    J20 Nationell>SHL (159, age 18) x SHL>NHL (19, age 21) = 0.058 [weight 19]

Published leagues method B can't reach: USHL


## 6. Stability: 2005-15 vs. 2016-25

Refit both methods on each half of the window separately (a pair counts in the half of its season t). If a factor moves a lot, either the league changed or the sample is too small to trust.

In [12]:
def factors_for(first, last):
    fa = method_a(next_season_pairs(careers, first, last)).set_index("league")
    fb = method_b(same_season_links(careers, first, last)).set_index("league")
    return fa, fb

a1, b1 = factors_for(2005, 2015)
a2, b2 = factors_for(2016, 2025)
stab = pd.DataFrame({
    "A 2005-15": a1["factor"], "A 2016-25": a2["factor"],
    "B 2005-15": b1["factor"], "B 2016-25": b2["factor"],
}).reindex(MAIN)
stab["A change"] = stab["A 2016-25"] / stab["A 2005-15"] - 1
stab["B change"] = stab["B 2016-25"] / stab["B 2005-15"] - 1
stab["A types"] = a1["path_type"].reindex(MAIN).str[0].fillna("-") + "/" + a2["path_type"].reindex(MAIN).str[0].fillna("-")
stab.round(3)

,A 2005-15,A 2016-25,B 2005-15,B 2016-25,A change,B change,A types
league,,,,,,,
KHL,0.770,0.613,0.571,0.456,-0.203,-0.201,d/d
SHL,0.610,0.481,0.493,0.357,-0.211,-0.276,d/d
NL,0.519,0.469,0.363,0.317,-0.096,-0.126,c/c
Liiga,0.516,0.356,0.323,0.264,-0.311,-0.182,c/c
AHL,0.467,0.464,0.396,0.367,-0.008,-0.074,d/d
DEL,0.396,0.446,0.206,0.284,0.127,0.379,c/c
HockeyAllsvenskan,0.395,0.341,0.173,0.131,-0.137,-0.243,c/c
NCAA,0.382,0.414,0.308,0.346,0.085,0.124,d/d
OHL,0.303,0.208,0.149,0.122,-0.314,-0.185,d/c


(`A types`: d = direct, c = chained, - = no factor, for the first and second half.)

## 7. Compared with published values

Published: hockeystats.com/methodology/nhle, built from every player in every league, 2005-2020. Any league more than 30% away is flagged, with the likely reason.

In [13]:
cmp = compare_published(a, b)

JUNIOR = {"OHL", "WHL", "QMJHL", "USHL", "MHL", "NCAA", "BCHL"}
def reason(lg, m):
    r = cmp.loc[lg]
    high = r[f"{m} vs pub"] > 0
    if m == "A" and high and lg in JUNIOR:
        why = "survivor sample: only the best jump straight to the NHL, published uses everyone"
        return why + ("; chained, so it also includes a year of development" if r["A type"] == "chained" else "")
    if m == "A" and high:
        return "only players good enough for an NHL job next season" + ("; chained" if r["A type"] == "chained" else "")
    if m == "B" and not high:
        return "same-season movers are often NHL players who struggled and left mid-season"
    if m == "B" and high:
        return "same-season NHL movers here are late-season signings, the best of their league"
    return ""

cmp["why A"] = [reason(lg, "A") if f == "FLAG" else "" for lg, f in cmp["A flag"].items()]
cmp["why B"] = [reason(lg, "B") if f == "FLAG" else "" for lg, f in cmp["B flag"].items()]
cmp.round(3)

,published,A,A type,B,B type,A vs pub,A flag,B vs pub,B flag,why A,why B
KHL,0.772,0.687,direct,0.619,direct,-0.110,,-0.198,,,
SHL,0.566,0.552,direct,0.335,direct,-0.025,,-0.409,FLAG,,same-season movers are often NHL players who struggled and left mid-season
NL,0.459,0.500,chained,0.292,direct,0.089,,-0.363,FLAG,,same-season movers are often NHL players who struggled and left mid-season
Liiga,0.441,0.431,chained,0.318,chained,-0.022,,-0.278,,,
AHL,0.389,0.466,direct,0.382,direct,0.198,,-0.017,,,
DEL,0.352,0.426,chained,0.301,chained,0.210,,-0.146,,,
HockeyAllsvenskan,0.351,0.377,chained,0.119,chained,0.074,,-0.660,FLAG,,same-season movers are often NHL players who struggled and left mid-season
NCAA,0.194,0.398,direct,0.339,direct,1.050,FLAG,0.745,FLAG,"survivor sample: only the best jump straight to the NHL, published uses everyone","same-season NHL movers here are late-season signings, the best of their league"
OHL,0.144,0.300,direct,0.149,direct,1.080,FLAG,0.037,,"survivor sample: only the best jump straight to the NHL, published uses everyone",
USHL,0.143,0.323,chained,NaN,NaN,1.261,FLAG,NaN,,"survivor sample: only the best jump straight to the NHL, published uses everyone; chained, so it also includes a year of development",


## 8. Backtest: which factors predict a rookie season best?

The real test: take each skater's **first NHL season with 20+ games** and predict his NHL points per game from the season right before it (the non-NHL league where he played the most games, 20+ games) × that league's factor.

- **Train on the past, test on the future.** Factors and age multipliers are fit only on data through 2017-18: method A pairs whose NHL season is 2017-18 or earlier, method B links from 2017-18 or earlier, rookies whose first NHL season is 2017-18 or earlier. The test is every rookie season from **2018-19 on**.
- **Versions:** A, B, and each with an **age multiplier**. Younger players improve faster, so an 18-year-old's junior scoring may be worth more than a 21-year-old's. The multiplier is one number per age bucket (age in the prior season): total actual NHL points / total predicted, on the training rookies.
- **Baseline:** call every rookie the training average for his position (F or D). Any NHLe version should beat that.
- **Error:** RMSE of points per game, weighted by NHL games. Scored on the **common sample** (rookies every version can predict), so the versions face the same players.

In [14]:
TRAIN_END = 2017
fa_train = method_a(next_season_pairs(careers, max_to_season=TRAIN_END)).set_index("league")
fb_train = method_b(same_season_links(careers, last=TRAIN_END)).set_index("league")

rookies = rookie_rows(careers)
train = rookies[rookies["season"] <= TRAIN_END]
test = rookies[rookies["season"] > TRAIN_END]
print(f"Rookie seasons (first NHL season with 20+ GP, prior season in the window): {len(rookies):,}")
print(f"Train: {label(train['season'].min())} to {label(train['season'].max())}, {len(train):,} rookies. "
      f"Test: {label(test['season'].min())} to {label(test['season'].max())}, {len(test):,} rookies")

mult = {}
for m, f in [("A", fa_train["factor"]), ("B", fb_train["factor"])]:
    mult[m] = age_multipliers(train, predict(train, f))
print("\nAge multipliers (fit on training rookies; 1.00 = the factor alone is right on average):")
pd.DataFrame(mult).T.round(2)

Rookie seasons (first NHL season with 20+ GP, prior season in the window): 1,587
Train: 2006-07 to 2017-18, 1,013 rookies. Test: 2018-19 to 2025-26, 574 rookies

Age multipliers (fit on training rookies; 1.00 = the factor alone is right on average):


bucket,<=18,19,20,21,22-24,25+
A,1.21,1.03,1.04,1.01,0.88,0.79
B,2.09,1.76,1.34,1.23,1.08,0.99


In [15]:
def all_versions(rows):
    """Predictions from every version for a set of rookie rows (training factors and multipliers)."""
    p = {
        "A": predict(rows, fa_train["factor"]),
        "A x age": predict(rows, fa_train["factor"], mult["A"]),
        "B": predict(rows, fb_train["factor"]),
        "B x age": predict(rows, fb_train["factor"], mult["B"]),
    }
    base = train.groupby("group").apply(lambda x: np.average(x["nhl_ppg"], weights=x["nhl_gp"]), include_groups=False)
    p["baseline (position average)"] = rows["group"].map(base)
    return p

def score(rows, preds, mask):
    out = {}
    for k, p in preds.items():
        sel = mask & p.notna()
        out[k] = {"RMSE": gp_rmse(rows["nhl_ppg"][sel], p[sel], rows["nhl_gp"][sel]),
                  "bias": gp_bias(rows["nhl_ppg"][sel], p[sel], rows["nhl_gp"][sel]),
                  "players": int(sel.sum())}
    return pd.DataFrame(out).T

preds = all_versions(test)
common = preds["A"].notna() & preds["B"].notna()
print(f"Test rookies: {len(test)}. Method A can predict {preds['A'].notna().sum()}, method B {preds['B'].notna().sum()}; "
      f"common sample {common.sum()}")

res = score(test, preds, common).rename(columns={"RMSE": "RMSE (common)", "bias": "bias (pred - actual)"})
res["RMSE F"] = score(test, preds, common & (test["group"] == "F"))["RMSE"]
res["RMSE D"] = score(test, preds, common & (test["group"] == "D"))["RMSE"]
own = pd.Series({k: gp_rmse(test["nhl_ppg"], p, test["nhl_gp"]) for k, p in preds.items()})
res["RMSE (own sample)"] = own
res["coverage"] = pd.Series({k: int(p.notna().sum()) for k, p in preds.items()})
VERSIONS = ["A", "A x age", "B", "B x age"]
winner = res.loc[VERSIONS, "RMSE (common)"].idxmin()
print(f"\nWinner: {winner}")
res.round(4)

Test rookies: 574. Method A can predict 574, method B 573; common sample 573

Winner: A x age


,RMSE (common),bias (pred - actual),players,RMSE F,RMSE D,RMSE (own sample),coverage
A,0.1482,0.0176,573.0,0.1602,0.1212,0.1487,574
A x age,0.1428,0.0098,573.0,0.1522,0.1222,0.1442,574
B,0.1591,-0.0653,573.0,0.1710,0.1328,0.1591,573
B x age,0.1473,0.0191,573.0,0.1595,0.1197,0.1473,573
baseline (position average),0.1651,-0.0120,573.0,0.1739,0.1462,0.1649,574


In [16]:
# Rookies whose prior league needed a chained method A factor (training factors)
chained_leagues = set(fa_train.index[fa_train["path_type"] == "chained"])
is_chained = test["prior_league"].isin(chained_leagues)
print(f"Test rookies from a chained league: {is_chained.sum()} "
      f"({', '.join(test.loc[is_chained, 'prior_league'].value_counts().index)})")
score(test, preds, is_chained).round(4)

Test rookies from a chained league: 26 (Liiga, QMJHL, DEL, NL, USHL, Czechia, HockeyAllsvenskan)


,RMSE,bias,players
A,0.1492,-0.0067,26.0
A x age,0.1562,0.0067,26.0
B,0.2011,-0.1504,25.0
B x age,0.1520,-0.0319,25.0
baseline (position average),0.1412,-0.0453,26.0


In [17]:
print("Where the test rookies came from (common sample):")
test.loc[common, "prior_league"].value_counts().head(12).to_frame("rookies").T

Where the test rookies came from (common sample):


prior_league,AHL,NCAA,SHL,KHL,WHL,OHL,Liiga,QMJHL,DEL,NL,Czechia,HockeyAllsvenskan
rookies,356,77,30,30,28,27,10,7,3,3,1,1


### Separate check: rookies sent down early (5-19 NHL games)

Not part of the main score. These are first NHL seasons (from 2018-19 on) with only 5 to 19 games: players who got a look and went back down. If the factors over-predict them (positive bias), that's expected. They were sent down partly because they didn't score.

In [18]:
short_rk = rookie_rows(careers, min_gp=5, max_gp=19)
short_rk = short_rk[short_rk["season"] > TRAIN_END]
short_preds = all_versions(short_rk)
short_common = short_preds["A"].notna() & short_preds["B"].notna()
short_res = score(short_rk, short_preds, short_common)
print(f"{len(short_rk)} short first seasons, {short_common.sum()} in the common sample")
for k in VERSIONS:
    b_ = short_res.loc[k, "bias"]
    print(f"  {k:8s} bias {b_:+.3f} points per game: " + ("over-predicts" if b_ > 0.01 else "under-predicts" if b_ < -0.01 else "about right"))
short_res.round(4)

343 short first seasons, 342 in the common sample
  A        bias +0.075 points per game: over-predicts
  A x age  bias +0.066 points per game: over-predicts
  B        bias +0.011 points per game: over-predicts
  B x age  bias +0.075 points per game: over-predicts


,RMSE,bias,players
A,0.2097,0.0747,342.0
A x age,0.2033,0.0664,342.0
B,0.1878,0.0114,342.0
B x age,0.2113,0.0754,342.0
baseline (position average),0.2179,0.1128,342.0


## 9. Hockey checks

Does the ordering make hockey sense? Roughly: KHL > SHL, Liiga and NL > AHL > NCAA > CHL and USHL, with the OHL, WHL and QMJHL close together and the QMJHL usually lowest. Printed, not enforced: a "no" is something to understand, not an error.

In [19]:
checks = ordering_checks(a.set_index("league")["factor"].to_dict()).rename(columns={"result": "method A"})
checks["method B"] = ordering_checks(b.set_index("league")["factor"].to_dict())["result"]
checks

,check,method A,method B
0,"KHL above SHL, Liiga and NL",yes,yes
1,"SHL, Liiga and NL above AHL",no,no
2,AHL above NCAA,yes,yes
3,"NCAA above OHL, WHL, QMJHL",yes,yes
4,NCAA above USHL,yes,n/a
5,"OHL, WHL, QMJHL within 25% of each other",yes,no
6,QMJHL lowest of the three CHL leagues,yes,yes


**Spot checks.** Recent rookies, with each method's full-window factors (section 4 and 5) and the winning version. The rookie's own season is one of hundreds in the fit, so this is a sanity check, not a test. Matthew Schaefer played only 17 OHL games in 2024-25 (injury), so the 20-game rule is waived for the spot checks. The European examples are Demidov (KHL), Dahlin (SHL) and Kakko (Liiga, a chained league). Lucas Raymond would have been one, but his last season before the NHL was 2020-21, which is skipped.

In [20]:
SPOT = {8484801: "Macklin Celebrini", 8483457: "Lane Hutson", 8485366: "Matthew Schaefer",
        8485402: "Michael Misa", 8484984: "Ivan Demidov", 8480839: "Rasmus Dahlin", 8481554: "Kaapo Kakko"}

fa_full, fb_full = a.set_index("league")["factor"], b.set_index("league")["factor"]
m_win, uses_age = winner.split()[0], "age" in winner
f_win = fa_full if m_win == "A" else fb_full
mult_full = age_multipliers(rookies, predict(rookies, f_win)) if uses_age else None

spot = rookie_rows(careers[careers["playerId"].isin(SPOT)], min_prior_gp=1)
spot = spot.assign(
    first_nhl=spot["season"].map(label), prior=spot["prior_league"] + " " + (spot["season"] - 1).map(label),
    factor_A=spot["prior_league"].map(fa_full), pred_A=predict(spot, fa_full),
    factor_B=spot["prior_league"].map(fb_full), pred_B=predict(spot, fb_full),
    pred_winner=predict(spot, f_win, mult_full))
cols = ["name", "prior", "prior_gp", "prior_ppg", "prior_age", "factor_A", "pred_A", "factor_B", "pred_B",
        "pred_winner", "first_nhl", "nhl_gp", "nhl_ppg"]
for pid, name in SPOT.items():
    if pid not in set(spot["playerId"]):
        print(f"{name}: no qualifying first NHL season (20+ GP) with a prior league season in the window")
print(f"pred_winner = {winner}")
spot[cols].round(3)

pred_winner = A x age


,name,prior,prior_gp,prior_ppg,prior_age,factor_A,pred_A,factor_B,pred_B,pred_winner,first_nhl,nhl_gp,nhl_ppg
0,Rasmus Dahlin,SHL 2017-18,41.0,0.488,17.0,0.552,0.269,0.335,0.163,0.301,2018-19,82.0,0.537
1,Kaapo Kakko,Liiga 2018-19,45.0,0.844,17.0,0.431,0.364,0.318,0.269,0.408,2019-20,66.0,0.348
2,Lane Hutson,NCAA 2023-24,38.0,1.289,19.0,0.398,0.513,0.339,0.437,0.539,2024-25,82.0,0.805
3,Macklin Celebrini,NCAA 2023-24,38.0,1.684,17.0,0.398,0.670,0.339,0.570,0.750,2024-25,70.0,0.900
4,Matthew Schaefer,OHL 2024-25,17.0,1.294,17.0,0.300,0.388,0.149,0.193,0.434,2025-26,82.0,0.720
5,Ivan Demidov,KHL 2024-25,65.0,0.754,19.0,0.687,0.518,0.619,0.467,0.545,2025-26,82.0,0.756
6,Michael Misa,OHL 2024-25,65.0,2.062,17.0,0.300,0.618,0.149,0.308,0.692,2025-26,45.0,0.467


## 10. Save the factors

Final factors use the **whole window** (2005-06 to 2025-26), both methods, with `chosen` marking the method that won the backtest. If the winner uses the age multiplier, the multipliers are refit on every rookie season and saved too.

In [21]:
factors = pd.concat([a.assign(method="A"), b.assign(method="B")], ignore_index=True)
factors["chosen"] = factors["method"] == m_win
factors = factors[["league", "method", "factor", "pairs", "path_type", "steps", "chosen", "paths"]]
factors.to_csv(OUT / "nhle_factors.csv", index=False)
print(f"Saved nhle_factors.csv: {len(factors)} rows ({(factors['method'] == 'A').sum()} method A, "
      f"{(factors['method'] == 'B').sum()} method B); chosen = method {m_win}")

if uses_age:
    mult_out = mult_full.rename("multiplier").rename_axis("age_bucket").reset_index()
    mult_out.to_csv(OUT / "nhle_age_multipliers.csv", index=False)
    print("Saved nhle_age_multipliers.csv:")
    print(mult_out.round(3).to_string(index=False))

Saved nhle_factors.csv: 94 rows (62 method A, 32 method B); chosen = method A
Saved nhle_age_multipliers.csv:
age_bucket  multiplier
      <=18       1.120
        19       1.052
        20       1.063
        21       0.995
     22-24       0.901
       25+       0.776


## Takeaways
- **Method A with the age multiplier wins the backtest.** On 573 rookie seasons from 2018-19 to 2025-26 (factors fit only on earlier years), its error is 0.143 points per game, vs. 0.147 for B with age, 0.148 for A alone, 0.159 for B alone and 0.165 for calling every rookie the position average (13% better than the baseline). It's best for forwards; for defensemen the four versions are within 0.003 of each other.
- **What that win means.** Method A is fit on players who jumped straight to the NHL, the same kind of player the backtest scores. So A winning means "best for projecting a prospect who makes the team," not "more accurate than published NHLe." Published values describe all players in each league, including the many who never reach the NHL.
- **The factors (method A, whole window):** KHL 0.69, SHL 0.55, AHL 0.47, NCAA 0.40, OHL 0.30, WHL 0.29, QMJHL 0.27 are direct. NL 0.50, Liiga 0.43, DEL 0.43, HockeyAllsvenskan 0.38, USHL 0.32, MHL 0.23 and BCHL 0.20 are chained through next-season moves.
- **Compared with published:** the European pro leagues and the AHL land within about 20% of published. Juniors and college run about double (NCAA +105%, OHL +108%, WHL +106%, QMJHL +135%, USHL +126%), because only the best of them jump straight to the NHL, and chained junior factors also carry a year of development. Method B's junior factors land close to published, but its European pro factors run low (SHL -41%, HockeyAllsvenskan -66%), because mid-season movers out of the NHL are often players who were struggling.
- **Age matters.** Junior or college scoring at 18 is worth about 12% more than the factor alone, and pro scoring at 25+ about 22% less. Younger players are still improving.
- **Not settled yet:**
  - **Chained leagues:** only 26 test rookies came from a chained league, and for them the position average did as well as any factor. The chains aren't proven, just not contradicted.
  - **Rookies sent down early** (5-19 games): every version over-predicts them (A with age by +0.07 points per game). That's expected, since they were sent down partly for not scoring.
  - **Stability:** KHL and SHL factors are about 20% lower in 2016-25 than in 2005-15 (Liiga 31%), while the AHL hasn't moved. European pro scoring is worth less now than it used to be, so whole-window factors may run a little high for those leagues.
- **Hockey checks:** the ordering makes sense apart from the AHL (0.47) ranking above Liiga (0.43) under method A. The spot checks are reasonable: Celebrini 0.75 predicted vs. 0.90 actual, Hutson 0.54 vs. 0.81, Demidov 0.55 vs. 0.76, Dahlin 0.30 vs. 0.54, Kakko 0.41 vs. 0.35, Misa 0.69 vs. 0.47, Schaefer 0.43 vs. 0.72.
- **Next:** development ranges (how far players beat or miss their projection), then a prospect rule that blends NHLe with any NHL games.